In [6]:
import os
os.environ["PATH"] += ":/opt/homebrew/bin"
pdf_dir = "/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis"
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files.")

Found 1 PDF files.


In [7]:
from langchain_community.document_loaders import UnstructuredPDFLoader

all_docs = []
for pdf_path in pdf_files:
    loader = UnstructuredPDFLoader(pdf_path,  # Adjust path if necessary
                                    mode="elements",  # Use

                                    extract_images_in_pdf=True,
        infer_table_structure=True)  # mode="elements" gives you text, tables, images, etc.
    docs = loader.load()
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} elements from all PDFs.")

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Loaded 1100 elements from all PDFs.


In [8]:
tables = [doc for doc in all_docs if doc.metadata.get("category") == "Table"]
images = [doc for doc in all_docs if doc.metadata.get("category") == "Image"]
texts = [doc for doc in all_docs if doc.metadata.get("category") in ["NarrativeText", "Title"]]

print(f"Texts: {len(texts)}, Tables: {len(tables)}, Images: {len(images)}")

Texts: 401, Tables: 47, Images: 40


In [9]:
table_texts = [doc.page_content for doc in tables]  # tables is your list of Document objects
#table_embeddings = embeddings.embed_documents(table_texts)

# Store in Milvus (append to your text chunks or as a separate collection)
# Example for appending:
#all_texts = texts_to_embed + table_texts

In [10]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

def get_image_embedding(image_path):
    image = Image.open(image_path)
    inputs = clip_processor(images=image, return_tensors="pt")
    outputs = clip_model.get_image_features(**inputs)
    return outputs.detach().numpy()[0]

image_paths = [doc.metadata["image_path"] for doc in images]  # images is your list of Document objects
image_embeddings = [get_image_embedding(path) for path in image_paths]

# Store image_embeddings and image_paths in Milvus (as a separate collection or together with text)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
!pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [11]:
from PyPDF2 import PdfReader

total_pages = 0
for pdf in pdf_files:
    reader = PdfReader(pdf)
    total_pages += len(reader.pages)
print(f"Total pages across all PDFs: {total_pages}")

Total pages across all PDFs: 77


## Semantic chunking over text

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

# Use a sentence-transformers model for embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = SentenceTransformer("all-MiniLM-L6-v2")
#Note: Use HuggingFaceEmbeddings instead of SentenceTransformer for compatibility with LangChain:
# Initialize the semantic chunker
chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=95)

# Suppose `texts` is your list of extracted text blocks from the PDFs
semantic_chunks = []
for text in texts:
    # This will return a list of semantically meaningful chunks for each text block
    semantic_chunks.extend(chunker.split_text(text.page_content))

print(f"Total semantic chunks: {len(semantic_chunks)}")

Total semantic chunks: 600


In [14]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

def get_image_embedding(image_path):
    image = Image.open(image_path)
    inputs = clip_processor(images=image, return_tensors="pt")
    outputs = clip_model.get_image_features(**inputs)
    return outputs.detach().numpy()[0]

image_paths = [doc.metadata["image_path"] for doc in images]  # images is your list of Document objects
image_embeddings = [get_image_embedding(path) for path in image_paths]

# Store image_embeddings and image_paths in Milvus (as a separate collection or together with text)

In [ ]:
## manually install pymilvus if not already done
# from pymilvus import Collection

# # Prepare your data
# all_embeddings = []
# all_contents = []
# all_modalities = []

# # Add text chunks
# for chunk in semantic_chunks:
#     all_embeddings.append(embeddings.embed_query(chunk))  # or use your batch embedding
#     all_contents.append(chunk)
#     all_modalities.append("text")

# # Add tables
# for table_text in table_texts:
#     all_embeddings.append(embeddings.embed_query(table_text))
#     all_contents.append(table_text)
#     all_modalities.append("table")

# # Add images
# for emb, path in zip(image_embeddings, image_paths):
#     all_embeddings.append(emb.tolist())  # ensure it's a list, not numpy array
#     all_contents.append(path)            # or base64 if you want to store image data
#     all_modalities.append("image")

# # Insert into Milvus
# collection = Collection("rag_chunks_flat")  # or your chosen collection
# entities = [
#     all_embeddings,
#     all_contents,
#     all_modalities
# ]
# collection.insert(entities)
# collection.flush()
# print("All modalities stored in Milvus!")

In [ ]:
# using langchain_community.vectorstores.Milvus to store the documents
# from langchain_community.vectorstores import Milvus
# from langchain_core.documents import Document

# docs = []
# for chunk in semantic_chunks:
#     docs.append(Document(page_content=chunk, metadata={"modality": "text"}))
# for table_text in table_texts:
#     docs.append(Document(page_content=table_text, metadata={"modality": "table"}))
# for path, emb in zip(image_paths, image_embeddings):
#     docs.append(Document(page_content=path, metadata={"modality": "image"}))

# milvus_db = Milvus.from_documents(
#     docs,
#     embedding=embeddings,
#     connection_args={
#         "host": "localhost",
#         "port": "19530",
#         "user": "",
#         "password": "",
#         "secure": False,
#     },
#     collection_name="rag_chunks_flat",
# )

In [20]:
# Cell 1: Create Milvus collections for each index type and insert data

from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from langchain_community.vectorstores import Milvus

from langchain_core.documents import Document
#connections.disconnect("default")  # Disconnect any previous connections
connections.connect(host="localhost", port="19530")
# Replace your connection with file-based Milvus Lite
# index type: HNSW, local mode only support FLAT IVF_FLAT AUTOINDEX:
#connections.connect("default", uri="./milvus_data.db")

EMBEDDING_DIM = 384  # Change if your embedding size is different

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=16535)
]
schema = CollectionSchema(fields, description="text and table chunks")

# Prepare your data
text_docs = []
text_docs_embeddings = []
for chunk in semantic_chunks:
    text_docs.append(Document(page_content=chunk, metadata={"modality": "text"}))
    text_docs_embeddings.append(embeddings.embed_query(chunk))  # Embed each chunk
for table_text in table_texts:
    text_docs.append(Document(page_content=table_text, metadata={"modality": "table"}))
    text_docs_embeddings.append(embeddings.embed_query(table_text))  # Embed each table text

# Embed the text chunks

text_embeddings = text_docs_embeddings
text_contents = [doc.page_content for doc in text_docs]

for index_type in ["FLAT", "HNSW", "IVF_FLAT"]:
    collection_name = f"rag_chunks_{index_type.lower()}"
    if utility.has_collection(collection_name):
        utility.drop_collection(collection_name)
    collection = Collection(name=collection_name, schema=schema)
    # Create index
    if index_type == "FLAT":
        index_params = {"index_type": "FLAT", "metric_type": "L2", "params": {}}
    elif index_type == "HNSW":
        index_params = {"index_type": "HNSW", "metric_type": "L2", "params": {"M": 8, "efConstruction": 64}}
    elif index_type == "IVF_FLAT":
        index_params = {"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}}
    collection.create_index(field_name="embedding", index_params=index_params)
    # Insert data
    entities = [
        text_embeddings,
        text_contents
    ]
    collection.insert(entities)
    collection.flush()
    print(f"Inserted data and created index: {collection_name} ({index_type})")

Inserted data and created index: rag_chunks_flat (FLAT)
Inserted data and created index: rag_chunks_hnsw (HNSW)
Inserted data and created index: rag_chunks_ivf_flat (IVF_FLAT)


In [21]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from langchain_community.vectorstores import Milvus

from langchain_core.documents import Document

connections.connect(host="localhost", port="19530")

EMBEDDING_DIM = 512  # Change if your embedding size is different

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=16535)
]
schema = CollectionSchema(fields, description="text and table chunks")

# Prepare your data
# text_docs = []
# text_docs_embeddings = []
# for chunk in semantic_chunks:
#     text_docs.append(Document(page_content=chunk, metadata={"modality": "text"}))
#     text_docs_embeddings.append(embeddings.embed_query(chunk))  # Embed each chunk
# for table_text in table_texts:
#     text_docs.append(Document(page_content=table_text, metadata={"modality": "table"}))
#     text_docs_embeddings.append(embeddings.embed_query(table_text))  # Embed each table text

# Embed the text chunks

text_embeddings = text_docs_embeddings
text_contents = [doc.page_content for doc in text_docs]

for index_type in ["FLAT", "HNSW", "IVF_FLAT"]:
    collection_name = f"rag_image_chunks_{index_type.lower()}"
    if utility.has_collection(collection_name):
        utility.drop_collection(collection_name)
    collection = Collection(name=collection_name, schema=schema)
    # Create index
    if index_type == "FLAT":
        index_params = {"index_type": "FLAT", "metric_type": "L2", "params": {}}
    elif index_type == "HNSW":
        index_params = {"index_type": "HNSW", "metric_type": "L2", "params": {"M": 8, "efConstruction": 64}}
    elif index_type == "IVF_FLAT":
        index_params = {"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}}
    collection.create_index(field_name="embedding", index_params=index_params)
    # Insert data
   # Insert image embeddings and paths
    entities = [
        image_embeddings,  # list of lists (vectors)
        image_paths        # list of strings
    ]
    collection.insert(entities)
    collection.flush()
    print("Image embeddings stored in Milvus!")
    print(f"Inserted data and created index: {collection_name} ({index_type})")

Image embeddings stored in Milvus!
Inserted data and created index: rag_image_chunks_flat (FLAT)
Image embeddings stored in Milvus!
Inserted data and created index: rag_image_chunks_hnsw (HNSW)
Image embeddings stored in Milvus!
Inserted data and created index: rag_image_chunks_ivf_flat (IVF_FLAT)


In [ ]:
# !docker run -d --name milvus-standalone  -p 19530:19530 -p 9091:9091 \
#   milvusdb/milvus:v2.4.0-rc.1  /bin/bash -c "milvus run standalone"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Unable to find image 'milvusdb/milvus:v2.4.0-rc.1' locally
v2.4.0-rc.1: Pulling from milvusdb/milvus

999ec938: Pulling fs layer 
aaa02da3: Pulling fs layer 
35a2405b: Pulling fs layer 
fe54d9b5: Pulling fs layer 
b700ef54: Pulling fs layer 
fe54d9b5: Downloading  49.28MB/72.38MBDownloading  23.07MB/27.36MBDownloading  3.146MB/72.38MBDownloading  22.02MB/51.97MBDownloading  23.07MB/51.97MBDownloading  25.17MB/51.97MBDownloading  5.243MB/72.38MBDownloading  6.291MB/72.38MBDownloading  6.291MB/72.38MBDownloading  355.5MB/426.9MBDownloading  363.9MB/426.9MBDownloading   7.34MB/72.38MBDownloading  405.8MB/426.9MBDownloading  416.3MB/426.9MBDownloading  9.437MB/72.38MBDownloading  9.437MB/72.38MBDownloading  9.437MB/72.38MBDownloading  9.437MB/72.38MBDownloading  9.437MB/72.38MBDownloading  10.49MB/72.38MBDownloading  10.49MB/72.38MBDownloading  11.53MB/72.38MBDownloading  11.53MB/72.38MBDownloading  11.53MB/72.38MBDownloading  12.58MB/72.38MBDownloading  13.63MB/72.38MBDownloading  13.63MB

In [41]:
from langchain_community.vectorstores import Milvus
from langchain_huggingface import HuggingFaceEmbeddings

# Assume semantic_chunks is a list of strings (your chunks)
# If you have Document objects, use [doc.page_content for doc in semantic_chunks]
texts_to_embed = semantic_chunks

# # Create embeddings
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Connect to local Milvus
milvus_db = Milvus.from_texts(
    texts=texts_to_embed,
    embedding=embeddings,
    connection_args={
        "host": "localhost",
        "port": "19530",
        "user": "",
        "password": "",
        "secure": False,
    },
    collection_name="rag_chunks1_flat",  # Use a unique collection name
)

print("Chunks stored in Milvus!")

Chunks stored in Milvus!


In [44]:
# Store images
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# Connect to Milvus
connections.connect(host="localhost", port="19530")

# Define your image embedding dimension (e.g., CLIP is 512)
IMAGE_EMBEDDING_DIM = 512

# Define fields for the image collection
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=IMAGE_EMBEDDING_DIM),
    FieldSchema(name="image_path", dtype=DataType.VARCHAR, max_length=512),
]
schema = CollectionSchema(fields, description="Image embeddings")

collection_name = "rag_chunks_flat_image"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
collection = Collection(name=collection_name, schema=schema)

# Insert image embeddings and paths
entities = [
    image_embeddings,  # list of lists (vectors)
    image_paths        # list of strings
]
collection.insert(entities)
collection.flush()
print("Image embeddings stored in Milvus!")

Image embeddings stored in Milvus!


In [29]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# Connect to Milvus
connections.connect(host="localhost", port="19530")

# Define your embedding dimension
EMBEDDING_DIM = 384  # or whatever your model outputs

# Define fields
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1024)
]
schema = CollectionSchema(fields, description="RAG chunks")

# Create collections for each index type
for index_type in ["FLAT", "HNSW", "IVF_FLAT"]:
    collection_name = f"rag_chunks_{index_type.lower()}"
    if utility.has_collection(collection_name):
        utility.drop_collection(collection_name)
    collection = Collection(name=collection_name, schema=schema)
    # Create index
    if index_type == "FLAT":
        index_params = {"index_type": "FLAT", "metric_type": "L2", "params": {}}
    elif index_type == "HNSW":
        index_params = {"index_type": "HNSW", "metric_type": "L2", "params": {"M": 8, "efConstruction": 64}}
    elif index_type == "IVF_FLAT":
        index_params = {"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}}
    collection.create_index(field_name="embedding", index_params=index_params)
    print(f"Created collection and index: {collection_name} ({index_type})")

Created collection and index: rag_chunks_flat (FLAT)
Created collection and index: rag_chunks_hnsw (HNSW)
Created collection and index: rag_chunks_ivf_flat (IVF_FLAT)


In [32]:

retriever = milvus_db.as_retriever(search_kwargs={"k": 5})

In [32]:
from langchain_community.vectorstores import Milvus
from langchain_huggingface import HuggingFaceEmbeddings
import time
# 1. Prepare embedding models
from pymilvus import Collection
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

connections.connect(host="localhost", port="19530")


#text_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

query = "What is the architecture of Llama 2?"
# 3. Get text embedding for text collection
text_emb = embeddings.embed_query(query)

# The original code tries to loop over variable names, but you want to loop over collection names as strings.
collection_names = [
    "rag_chunks_ivf_flat",
    "rag_chunks_flat",
    "rag_chunks_hnsw"
]

for collection_name in collection_names:
    try:
        print(f"Processing {collection_name}")

        if not utility.has_collection(collection_name):
            print(f"Collection {collection_name} does not exist. Skipping.")
            continue

        start_time = time.time()
        milvus_text = Milvus(
            collection_name=collection_name,
            embedding_function=embeddings,
            connection_args={
                "host": "localhost",
                "port": "19530",
                "user": "",
                "password": "",
                "secure": False,
            },
            vector_field="embedding",
        )
    except KeyError as e:
        print(f"KeyError occurred with collection_name={collection_name}: {e}")

    # 4. Retrieve relevant documents
    text_retriever = milvus_text.as_retriever(search_kwargs={"k": 5})
    text_results = text_retriever.get_relevant_documents(query)
    end_time = time.time()
    ptime_taken = end_time - start_time
    print(f"Time taken for {collection_name}: {ptime_taken:.2f} seconds")

Processing rag_chunks_ivf_flat
Time taken for rag_chunks_ivf_flat: 1.45 seconds
Processing rag_chunks_flat
KeyError occurred with collection_name=rag_chunks_flat: 'FLAT'
Time taken for rag_chunks_flat: 0.02 seconds
Processing rag_chunks_hnsw
Time taken for rag_chunks_hnsw: 1.44 seconds


Bad pipe message: %s [b'\\\xe4\x86g\x99\xd5\xd6A\xf0\xc0']
Bad pipe message: %s [b'\x8b\x87{\xadD\xe6\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x96\x00', b"\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xba\x00\xbb\x00\xbc\x00\xbd\x00\xbe\x00\xbf\x00\xc0\x00\xc1\x00\xc2\x00\xc3\x00\xc4\x00\xc5\x13\x01\x13\x02\x13\x03\x13\x04\x13\x05\xc0\x01\xc0\x02\xc0\x03\xc0\x04\xc0\x05\xc0\x06\xc0\x07\xc0\x08\xc0\t\xc0\n\xc0\x0b\xc0\x0c\xc0\r\xc0\x0e\xc0\x0f\xc0\x10\xc0\x11\xc0\x12\xc0\x13\xc0\x14\xc0\x15\xc0\x16\xc0\x17\xc0\x18\xc0\x

In [24]:
from pymilvus import utility, connections

connections.connect(host="localhost", port="19530")
collections = utility.list_collections()
print("Collections in Milvus:")
for name in collections:
    print(name)

Collections in Milvus:
rag_image_chunks_ivf_flat
rag_chunks_flat_image
rag_chunks_ivf_flat
rag_image_chunks_flat
rag_chunks1_flat
rag_chunks_flat
rag_chunks
rag_chunks_hnsw
rag_image_chunks_hnsw
